In [ ]:
# Liquor Store Sales & Inventory Analytics

In [3]:
import os
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

In [ ]:
data_path = Path(r"D:\data")

print("Data folder exists:", data_path.exists())
print("Data folder:", data_path)

In [ ]:
csv_files = list(data_path.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file.name)

In [ ]:
#CHECK SIZE OF EVERY FILE

for file in csv_files:
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name:25} {size_mb:10.2f} MB")

In [ ]:
# REUSABLE FUNCTION
import csv

def get_columns(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        reader = csv.reader(file)
        return next(reader)

In [ ]:
for file in csv_files:
    columns = get_columns(file)
    
    print("\n" + "=" * 60)
    print(file.name)
    print("=" * 60)
    
    print("Number of columns:", len(columns))
    print("Columns:")
    
    for i, column in enumerate(columns, start=1):
        print(f"{i}. {column}")

In [ ]:
def preview_csv(file_path, n=5):
    with open(file_path, "r", encoding="utf-8") as file:
        reader = csv.reader(file)
        
        for _ in range(n):
            try:
                print(next(reader))
            except StopIteration:
                break

In [ ]:
for file in csv_files:
    print("\n" + "=" * 80)
    print(file.name)
    print("=" * 80)
    
    preview_csv(file, 5)

In [ ]:
def count_rows(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return sum(1 for _ in file) - 1

In [ ]:
for file in csv_files:
    rows = count_rows(file)
    print(f"{file.name:25} {rows:,} rows")

In [ ]:
dataset_summary = []

for file in csv_files:
    columns = get_columns(file)
    rows = count_rows(file)
    size_mb = file.stat().st_size / (1024 ** 2)
    
    dataset_summary.append({
        "file": file.name,
        "rows": rows,
        "columns": len(columns),
        "size_mb": round(size_mb, 2)
    })

dataset_summary

In [ ]:
for file in csv_files:
    columns = get_columns(file)

    print("\n" + "=" * 70)
    print(file.name)
    print("=" * 70)

    for i, column in enumerate(columns, start=1):
        print(f"{i:2}. {column}")

In [ ]:
purchase_prices_file = data_path / "purchase_prices.csv"

In [ ]:
purchase_prices = pd.read_csv(purchase_prices_file)

In [ ]:
purchase_prices.head()

In [ ]:
purchase_prices.shape

In [ ]:
purchase_prices.columns

In [ ]:
purchase_prices.info()

In [ ]:
purchase_prices.describe()

In [ ]:
purchase_prices["Brand"].nunique()

In [ ]:
purchase_prices["VendorNumber"].nunique()

In [ ]:
purchase_prices["VendorName"].nunique()

In [ ]:
purchase_prices["Brand"].value_counts().head(10)

In [ ]:
vendor_product_count = (
    purchase_prices
    .groupby("VendorName")["Brand"]
    .nunique()
    .sort_values(ascending=False)
)

vendor_product_count.head(10)

In [ ]:
sales_file = data_path / "sales.csv"

print(sales_file)
print(f"{sales_file.stat().st_size / (1024**3):.2f} GB")

In [ ]:
sales_sample = pd.read_csv(
    sales_file,
    nrows=10_000
)

sales_sample.head()

In [ ]:
sales_sample.shape

In [ ]:
sales_sample.info()

In [ ]:
sales_sample.dtypes

In [ ]:
chunk_size = 200_000

sales_reader = pd.read_csv(
    sales_file,
    chunksize=chunk_size
)

In [ ]:
first_chunk = next(sales_reader)

print(first_chunk.shape)

In [ ]:
total_rows = 0

for chunk in pd.read_csv(
    sales_file,
    chunksize=200_000
):
    total_rows += len(chunk)

print(f"Total sales records: {total_rows:,}")

In [ ]:
total_sales = 0

for chunk in pd.read_csv(
    sales_file,
    chunksize=200_000
):
    total_sales += chunk["SalesDollars"].sum()

print(f"Total sales revenue: ${total_sales:,.2f}")

In [ ]:
total_units = 0

for chunk in pd.read_csv(
    sales_file,
    chunksize=200_000
):
    total_units += chunk["SalesQuantity"].sum()

print(f"Total units sold: {total_units:,.0f}")

In [ ]:
total_revenue = 0
total_quantity = 0

for chunk in pd.read_csv(
    sales_file,
    chunksize=200_000
):
    total_revenue += chunk["SalesDollars"].sum()
    total_quantity += chunk["SalesQuantity"].sum()

average_selling_price = total_revenue / total_quantity

print(f"Average selling price: ${average_selling_price:.2f}")

In [ ]:
total_revenue = 0
total_quantity = 0

for chunk in pd.read_csv(
    sales_file,
    chunksize=200_000
):
    total_revenue += chunk["SalesDollars"].sum()
    total_quantity += chunk["SalesQuantity"].sum()

average_selling_price = total_revenue / total_quantity

print(f"Average selling price: ${average_selling_price:.2f}")

In [ ]:
min_date = None
max_date = None

for chunk in pd.read_csv(
    sales_file,
    chunksize=200_000
):
    dates = pd.to_datetime(
        chunk["SalesDate"],
        errors="coerce"
    )

    chunk_min = dates.min()
    chunk_max = dates.max()

    if min_date is None or chunk_min < min_date:
        min_date = chunk_min

    if max_date is None or chunk_max > max_date:
        max_date = chunk_max

print("Earliest sale:", min_date)
print("Latest sale:", max_date)

In [ ]:
missing_counts = None

for chunk in pd.read_csv(
    sales_file,
    chunksize=200_000
):
    current_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = current_missing
    else:
        missing_counts += current_missing

missing_counts

In [ ]:
missing_percentage = (
    missing_counts / total_rows * 100
).sort_values(ascending=False)

missing_percentage

In [ ]:
sales_profile = {
    "rows": total_rows,
    "total_revenue": total_revenue,
    "total_units": total_quantity,
    "average_selling_price": average_selling_price,
    "min_date": min_date,
    "max_date": max_date
}

sales_profile

In [ ]:
import pandas as pd

sales_file = data_path / "sales.csv"

CHUNK_SIZE = 200_000

total_rows = 0
total_revenue = 0
total_units = 0

min_date = None
max_date = None

missing_counts = None

stores = set()
brands = set()
vendors = set()
classifications = set()

for chunk in pd.read_csv(sales_file, chunksize=CHUNK_SIZE):

    
    total_rows += len(chunk)

    
    total_revenue += chunk["SalesDollars"].sum()
    total_units += chunk["SalesQuantity"].sum()

    
    dates = pd.to_datetime(chunk["SalesDate"], errors="coerce")

    chunk_min = dates.min()
    chunk_max = dates.max()

    if min_date is None or chunk_min < min_date:
        min_date = chunk_min

    if max_date is None or chunk_max > max_date:
        max_date = chunk_max

    
    current_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = current_missing
    else:
        missing_counts += current_missing

    
    stores.update(chunk["Store"].dropna().unique())
    brands.update(chunk["Brand"].dropna().unique())
    vendors.update(chunk["VendorNo"].dropna().unique())
    classifications.update(chunk["Classification"].dropna().unique())


average_selling_price = total_revenue / total_units

sales_profile = {
    "Total Rows": total_rows,
    "Total Revenue": total_revenue,
    "Total Units Sold": total_units,
    "Average Selling Price": average_selling_price,
    "Earliest Sale": min_date,
    "Latest Sale": max_date,
    "Unique Stores": len(stores),
    "Unique Brands": len(brands),
    "Unique Vendors": len(vendors),
    "Unique Classifications": len(classifications)
}

sales_profile

In [ ]:
missing_report = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percentage": (missing_counts / total_rows * 100)
})

missing_report = (
    missing_report
    .sort_values("Missing_Percentage", ascending=False)
)

missing_report

In [ ]:
memory_usage = (
    sales_sample.memory_usage(deep=True)
    .sort_values(ascending=False)
)

memory_usage

In [ ]:
duplicate_rows = 0
total_rows = 0

for chunk in pd.read_csv(sales_file, chunksize=200_000):

    total_rows += len(chunk)
    duplicate_rows += chunk.duplicated().sum()

print("Total rows:", total_rows)
print("Duplicate rows found:", duplicate_rows)
print("Duplicate percentage:", duplicate_rows / total_rows * 100)

In [ ]:
numeric_columns = [
    "SalesQuantity",
    "SalesDollars",
    "SalesPrice",
    "Volume",
    "ExciseTax"
]

In [ ]:
numeric_quality = {
    column: {
        "negative": 0,
        "zero": 0,
        "missing": 0
    }
    for column in numeric_columns
}

for chunk in pd.read_csv(sales_file, chunksize=200_000):

    for column in numeric_columns:

        numeric_quality[column]["negative"] += (
            chunk[column] < 0
        ).sum()

        numeric_quality[column]["zero"] += (
            chunk[column] == 0
        ).sum()

        numeric_quality[column]["missing"] += (
            chunk[column].isna()
        ).sum()

In [ ]:
quality_report = pd.DataFrame(numeric_quality).T

quality_report

In [ ]:
sample = pd.read_csv(
    sales_file,
    nrows=100_000
)

sample["ExpectedSales"] = (
    sample["SalesQuantity"] * sample["SalesPrice"]
)

sample["SalesDifference"] = (
    sample["SalesDollars"] - sample["ExpectedSales"]
)

sample["SalesDifference"].describe()

In [ ]:
tolerance = 0.01

invalid_sales_calculation = (
    sample["SalesDifference"].abs() > tolerance
).sum()

invalid_percentage = (
    invalid_sales_calculation / len(sample) * 100
)

print("Rows with calculation difference:", invalid_sales_calculation)
print("Percentage:", invalid_percentage)

In [ ]:
sample["SalesDate"] = pd.to_datetime(
    sample["SalesDate"],
    errors="coerce"
)

print("Invalid dates:", sample["SalesDate"].isna().sum())
print("Minimum date:", sample["SalesDate"].min())
print("Maximum date:", sample["SalesDate"].max())

In [ ]:
future_dates = (
    sample["SalesDate"] > pd.Timestamp.today()
).sum()

print("Future dates:", future_dates)

In [ ]:
categorical_columns = [
    "Store",
    "Classification",
    "VendorName"
]

In [ ]:
for column in categorical_columns:

    print(f"\n--- {column} ---")
    print("Unique values:", sample[column].nunique())
    print(sample[column].value_counts(dropna=False).head(15))

In [ ]:
data_quality_summary = {
    "total_rows": total_rows,
    "duplicate_rows": duplicate_rows,
    "duplicate_percentage": duplicate_rows / total_rows * 100,
    "invalid_dates_sample": sample["SalesDate"].isna().sum(),
    "future_dates_sample": future_dates,
    "sales_calculation_errors_sample": invalid_sales_calculation,
    "sales_calculation_error_percentage_sample": invalid_percentage
}

data_quality_summary

In [ ]:
def clean_sales_chunk(chunk):

    # 1. Remove exact duplicates
    chunk = chunk.drop_duplicates()

    # 2. Clean only actual text columns
    text_columns = [
        "Description",
        "VendorName"
    ]

    for column in text_columns:
        chunk[column] = chunk[column].astype("string").str.strip()

    # 3. Convert numeric columns
    numeric_columns = [
        "Store",
        "Brand",
        "Size",
        "SalesQuantity",
        "SalesDollars",
        "SalesPrice",
        "Volume",
        "Classification",
        "ExciseTax",
        "VendorNo"
    ]

    for column in numeric_columns:
        chunk[column] = pd.to_numeric(
            chunk[column],
            errors="coerce"
        )

    # 4. Convert SalesDate
    chunk["SalesDate"] = pd.to_datetime(
        chunk["SalesDate"],
        errors="coerce"
    )

    # 5. Remove rows missing critical fields
    critical_columns = [
        "SalesDate",
        "SalesQuantity",
        "SalesDollars",
        "SalesPrice"
    ]

    chunk = chunk.dropna(
        subset=critical_columns
    )

    return chunk

In [ ]:
numeric_columns = [
    "Store",
    "Brand",
    "Size",
    "SalesQuantity",
    "SalesDollars",
    "SalesPrice",
    "Volume",
    "ExciseTax",
    "VendorNo"
]

In [ ]:
cleaned_sales_file = data_path / "cleaned_sales.csv"

first_chunk = True
total_before = 0
total_after = 0

for chunk in pd.read_csv(
    sales_file,
    chunksize=200_000
):

    total_before += len(chunk)

    cleaned_chunk = clean_sales_chunk(chunk)

    total_after += len(cleaned_chunk)

    cleaned_chunk.to_csv(
        cleaned_sales_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

print("Cleaning completed.")
print("Rows before:", total_before)
print("Rows after:", total_after)
print("Rows removed:", total_before - total_after)

In [ ]:
cleaned_sample = pd.read_csv(
    cleaned_sales_file,
    nrows=100_000
)

cleaned_sample.info()

In [ ]:
cleaned_sample.isna().sum()

In [ ]:
print(
    "Duplicate rows:",
    cleaned_sample.duplicated().sum()
)

In [ ]:
cleaned_sample.dtypes

In [ ]:
cleaned_sample["SalesDate"] = pd.to_datetime(
    cleaned_sample["SalesDate"],
    errors="coerce"
)

print("Invalid dates:", cleaned_sample["SalesDate"].isna().sum())
print("Minimum date:", cleaned_sample["SalesDate"].min())
print("Maximum date:", cleaned_sample["SalesDate"].max())

In [ ]:
purchase_prices = pd.read_csv(
    data_path / "purchase_prices.csv"
)

print("Rows:", len(purchase_prices))
print("Unique Brands:", purchase_prices["Brand"].nunique())

print(
    "Duplicate Brand rows:",
    purchase_prices["Brand"].duplicated().sum()
)

In [ ]:
begin_inventory = pd.read_csv(
    data_path / "begin_inventory.csv"
)

print("Rows:", len(begin_inventory))
print(
    "Unique InventoryId:",
    begin_inventory["InventoryId"].nunique()
)

print(
    "Duplicate InventoryId:",
    begin_inventory["InventoryId"].duplicated().sum()
)

In [ ]:
vendor_invoice = pd.read_csv(
    data_path / "vendor_invoice.csv"
)

print("Vendor rows:", len(vendor_invoice))

print(
    "Unique VendorNumber:",
    vendor_invoice["VendorNumber"].nunique()
)

print(
    "Duplicate VendorNumber:",
    vendor_invoice["VendorNumber"].duplicated().sum()
)

print(
    "Unique PONumber:",
    vendor_invoice["PONumber"].nunique()
)

print(
    "Duplicate PONumber:",
    vendor_invoice["PONumber"].duplicated().sum()
)

In [ ]:

begin_inventory = pd.read_csv(data_path / "begin_inventory.csv")
end_inventory = pd.read_csv(data_path / "end_inventory.csv")
purchase_prices = pd.read_csv(data_path / "purchase_prices.csv")
vendor_invoice = pd.read_csv(data_path / "vendor_invoice.csv")

print("========== BEGIN INVENTORY ==========")
print("Rows:", len(begin_inventory))
print("Unique InventoryId:", begin_inventory["InventoryId"].nunique())
print("Duplicate InventoryId:", begin_inventory["InventoryId"].duplicated().sum())

print("\n========== END INVENTORY ==========")
print("Rows:", len(end_inventory))
print("Unique InventoryId:", end_inventory["InventoryId"].nunique())
print("Duplicate InventoryId:", end_inventory["InventoryId"].duplicated().sum())

print("\n========== PURCHASE PRICES ==========")
print("Rows:", len(purchase_prices))
print("Unique Brand:", purchase_prices["Brand"].nunique())
print("Duplicate Brand:", purchase_prices["Brand"].duplicated().sum())

print("\n========== VENDORS ==========")
print("Rows:", len(vendor_invoice))
print("Unique VendorNumber:", vendor_invoice["VendorNumber"].nunique())
print("Duplicate VendorNumber:",
      vendor_invoice["VendorNumber"].duplicated().sum())

print("\n========== PURCHASE ORDERS ==========")
print("Rows:", len(vendor_invoice))
print("Unique PONumber:", vendor_invoice["PONumber"].nunique())
print("Duplicate PONumber:",
      vendor_invoice["PONumber"].duplicated().sum())

In [ ]:
purchases_sample = pd.read_csv(
    data_path / "purchases.csv",
    nrows=100_000
)

purchase_po = set(purchases_sample["PONumber"].dropna().unique())
invoice_po = set(vendor_invoice["PONumber"].dropna().unique())

print("Purchase POs:", len(purchase_po))
print("Invoice POs:", len(invoice_po))
print("Matching POs:", len(purchase_po & invoice_po))
print("Purchase POs without invoice:",
      len(purchase_po - invoice_po))

In [ ]:
begin_inventory = pd.read_csv(
    data_path / "begin_inventory.csv"
)

end_inventory = pd.read_csv(
    data_path / "end_inventory.csv"
)

purchases = pd.read_csv(
    data_path / "purchases.csv"
)

purchase_prices = pd.read_csv(
    data_path / "purchase_prices.csv"
)

vendor_invoice = pd.read_csv(
    data_path / "vendor_invoice.csv"
)


print("=" * 70)
print("1. PRODUCT / BRAND VALIDATION")
print("=" * 70)

brand_description = (
    purchase_prices
    .groupby("Brand")["Description"]
    .nunique()
)

brand_size = (
    purchase_prices
    .groupby("Brand")["Size"]
    .nunique()
)

brand_classification = (
    purchase_prices
    .groupby("Brand")["Classification"]
    .nunique()
)

print("Total rows:", len(purchase_prices))
print("Unique brands:", purchase_prices["Brand"].nunique())

print(
    "Brands with multiple descriptions:",
    (brand_description > 1).sum()
)

print(
    "Brands with multiple sizes:",
    (brand_size > 1).sum()
)

print(
    "Brands with multiple classifications:",
    (brand_classification > 1).sum()
)



In [ ]:

print("\n" + "=" * 70)
print("2. VENDOR VALIDATION")
print("=" * 70)

vendor_names = (
    vendor_invoice
    .groupby("VendorNumber")["VendorName"]
    .nunique()
)

print("Unique vendors:", vendor_invoice["VendorNumber"].nunique())

print(
    "Vendors with multiple names:",
    (vendor_names > 1).sum()
)


In [ ]:
print("\n" + "=" * 70)
print("3. STORE VALIDATION")
print("=" * 70)

store_city = (
    begin_inventory
    .groupby("Store")["City"]
    .nunique()
)

print("Unique stores:", begin_inventory["Store"].nunique())

print(
    "Stores with multiple cities:",
    (store_city > 1).sum()
)



In [ ]:
print("\n" + "=" * 70)
print("4. INVENTORY ID VALIDATION")
print("=" * 70)

print(
    "Begin inventory rows:",
    len(begin_inventory)
)

print(
    "Begin unique InventoryId:",
    begin_inventory["InventoryId"].nunique()
)

print(
    "Begin duplicate InventoryId:",
    begin_inventory["InventoryId"].duplicated().sum()
)

print(
    "End inventory rows:",
    len(end_inventory)
)

print(
    "End unique InventoryId:",
    end_inventory["InventoryId"].nunique()
)

print(
    "End duplicate InventoryId:",
    end_inventory["InventoryId"].duplicated().sum()
)




In [ ]:
print("\n" + "=" * 70)
print("5. PURCHASE ORDER VALIDATION")
print("=" * 70)

po_counts = (
    purchases
    .groupby("PONumber")
    .size()
)

print(
    "Purchase rows:",
    len(purchases)
)

print(
    "Unique purchase orders:",
    purchases["PONumber"].nunique()
)

print(
    "Duplicate PONumber rows:",
    purchases["PONumber"].duplicated().sum()
)

print(
    "POs containing multiple purchase lines:",
    (po_counts > 1).sum()
)




In [ ]:
print("\n" + "=" * 70)
print("6. PRODUCT → VENDOR RELATIONSHIP")
print("=" * 70)

product_vendor = (
    purchase_prices
    .groupby("Brand")["VendorNumber"]
    .nunique()
)

print(
    "Brands:",
    len(product_vendor)
)

print(
    "Brands with multiple vendors:",
    (product_vendor > 1).sum()
)



In [ ]:

print("\n" + "=" * 70)
print("7. INVENTORY ID RELATIONSHIP")
print("=" * 70)

inventory_store = (
    begin_inventory
    .groupby("InventoryId")["Store"]
    .nunique()
)

inventory_brand = (
    begin_inventory
    .groupby("InventoryId")["Brand"]
    .nunique()
)

print(
    "Inventory IDs with multiple stores:",
    (inventory_store > 1).sum()
)

print(
    "Inventory IDs with multiple brands:",
    (inventory_brand > 1).sum()
)



In [ ]:

print("\n" + "=" * 70)
print("8. PO → VENDOR RELATIONSHIP")
print("=" * 70)

po_vendor = (
    purchases
    .groupby("PONumber")["VendorNumber"]
    .nunique()
)

print(
    "POs with multiple vendors:",
    (po_vendor > 1).sum()
)


In [ ]:
from pathlib import Path

data_path = Path(r"D:\data")

purchases = pd.read_csv(
    data_path / "purchases.csv",
    usecols=["VendorNumber", "VendorName"]
)

vendor_check = (
    purchases
    .drop_duplicates()
    .groupby("VendorNumber")["VendorName"]
    .agg(list)
)

conflicting_vendors = vendor_check[
    vendor_check.apply(len) > 1
]

print("Vendors with conflicting names:", len(conflicting_vendors))
print()

for vendor_number, names in conflicting_vendors.items():
    print(f"VendorNumber: {vendor_number}")
    for name in names:
        print(f"   - {repr(name)}")
    print()

In [ ]:
data_path = Path(r"D:\data")

files = [
    "begin_inventory.csv",
    "end_inventory.csv",
    "purchases.csv",
    "purchase_prices.csv",
    "vendor_invoice.csv",
    "sales.csv"
]

for file in files:
    print("\n" + "=" * 80)
    print(file.upper())
    print("=" * 80)

    
    df = pd.read_csv(
        data_path / file,
        nrows=10000
    )

    print("\nColumns + Pandas datatype:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nUnique values for low-cardinality columns:")
    for col in df.columns:
        if df[col].nunique(dropna=True) <= 15:
            print(f"{col}: {df[col].dropna().unique()[:15]}")

In [ ]:
DB_USER = "postgres"
DB_PASSWORD = "Harshit29"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "liquor_analytics"


connection_string = (
    f"postgresql+psycopg2://"
    f"{DB_USER}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/"
    f"{DB_NAME}"
)

engine = create_engine(connection_string)

print("PostgreSQL connection object created successfully.")


test_query = """
SELECT
    current_database() AS database_name,
    current_user AS username,
    version() AS postgres_version;
"""

connection_test = pd.read_sql(test_query, engine)

display(connection_test)


daily_kpi = pd.read_sql(
    """
    SELECT *
    FROM analytics.vw_daily_kpi
    ORDER BY sales_date;
    """,
    engine
)

print("Rows:", len(daily_kpi))
print("Columns:", daily_kpi.columns.tolist())

display(daily_kpi.head())



In [ ]:
print(daily_kpi.dtypes)


# Convert date explicitly

daily_kpi["sales_date"] = pd.to_datetime(
    daily_kpi["sales_date"],
    errors="coerce"
)

print("\nInvalid dates:")
print(daily_kpi["sales_date"].isna().sum())


print("Shape:")
print(daily_kpi.shape)

print("\nMissing values:")
print(daily_kpi.isna().sum())

print("\nDuplicate rows:")
print(daily_kpi.duplicated().sum())

print("\nStatistical summary:")
display(daily_kpi.describe())



In [ ]:
print("==========================================")
print("BUSINESS KPI SUMMARY")
print("==========================================")

total_revenue = daily_kpi["revenue"].sum()

total_units = daily_kpi["units_sold"].sum()

total_sales_lines = daily_kpi["transaction_lines"].sum()

average_price = daily_kpi["avg_selling_price"].mean()

total_excise_tax = daily_kpi["excise_tax"].sum()

print("Total Revenue:", round(total_revenue, 2))
print("Total Units Sold:", total_units)
print("Total Sales Lines:", total_sales_lines)
print("Average Selling Price:", round(average_price, 2))
print("Total Excise Tax:", round(total_excise_tax, 2))



In [ ]:
revenue_array = daily_kpi["revenue"].to_numpy()

print("Revenue Mean   :", np.mean(revenue_array))
print("Revenue Median :", np.median(revenue_array))
print("Revenue Std    :", np.std(revenue_array))
print("Revenue Min    :", np.min(revenue_array))
print("Revenue Max    :", np.max(revenue_array))



In [ ]:
daily_kpi["previous_day_revenue"] = (
    daily_kpi["revenue"].shift(1)
)

daily_kpi["revenue_change"] = (
    daily_kpi["revenue"]
    - daily_kpi["previous_day_revenue"]
)

daily_kpi["revenue_growth_percent"] = (
    daily_kpi["revenue_change"]
    / daily_kpi["previous_day_revenue"]
    * 100
)

display(
    daily_kpi[
        [
            "sales_date",
            "revenue",
            "previous_day_revenue",
            "revenue_change",
            "revenue_growth_percent"
        ]
    ].head(10)
)




In [ ]:
# ============================================================
# BEST AND WORST SALES DAYS
# ============================================================

best_day = daily_kpi.loc[
    daily_kpi["revenue"].idxmax()
]

worst_day = daily_kpi.loc[
    daily_kpi["revenue"].idxmin()
]

print("BEST SALES DAY")
print(best_day)

print("\nWORST SALES DAY")
print(worst_day)


# ============================================================
#  LOAD PRODUCT SALES
# ============================================================

product_sales = pd.read_sql(
    """
    SELECT *
    FROM analytics.vw_product_sales;
    """,
    engine
)

print("Product rows:", len(product_sales))

display(product_sales.head())


# ============================================================
#  TOP 20 PRODUCTS
# ============================================================

top_products = (
    product_sales
    .sort_values("revenue", ascending=False)
    .head(20)
)

display(
    top_products[
        [
            "brand_id",
            "description",
            "units_sold",
            "revenue"
        ]
    ]
)




In [ ]:
# ============================================================
#  PRODUCT REVENUE CONTRIBUTION
# ============================================================

product_sales["revenue_percent"] = (
    product_sales["revenue"]
    / product_sales["revenue"].sum()
    * 100
)

product_sales = product_sales.sort_values(
    "revenue",
    ascending=False
)

product_sales["cumulative_revenue_percent"] = (
    product_sales["revenue_percent"].cumsum()
)

display(
    product_sales[
        [
            "brand_id",
            "description",
            "revenue",
            "revenue_percent",
            "cumulative_revenue_percent"
        ]
    ].head(20)
)


# ============================================================
#  ABC CLASSIFICATION
# ============================================================

product_sales["abc_category"] = np.select(
    [
        product_sales["cumulative_revenue_percent"] <= 80,
        product_sales["cumulative_revenue_percent"] <= 95
    ],
    [
        "A",
        "B"
    ],
    default="C"
)

display(
    product_sales[
        [
            "brand_id",
            "description",
            "revenue",
            "abc_category"
        ]
    ].head(30)
)


# ============================================================
#  ABC SUMMARY
# ============================================================

abc_summary = (
    product_sales
    .groupby("abc_category")
    .agg(
        products=("brand_id", "count"),
        revenue=("revenue", "sum"),
        units=("units_sold", "sum")
    )
    .reset_index()
)

abc_summary["revenue_percent"] = (
    abc_summary["revenue"]
    / abc_summary["revenue"].sum()
    * 100
)

display(abc_summary)


# ============================================================
# LOAD STORE SALES
# ============================================================

store_sales = pd.read_sql(
    """
    SELECT *
    FROM analytics.vw_store_sales;
    """,
    engine
)

print("Stores:", len(store_sales))

display(store_sales.head())


# ============================================================
# TOP STORES
# ============================================================

top_stores = (
    store_sales
    .sort_values("revenue", ascending=False)
    .head(10)
)

display(top_stores)


# ============================================================
# STORE REVENUE SHARE
# ============================================================

store_sales["revenue_share_percent"] = (
    store_sales["revenue"]
    / store_sales["revenue"].sum()
    * 100
)

display(
    store_sales
    .sort_values(
        "revenue",
        ascending=False
    )
    .head(20)
)


# ============================================================
# LOAD VENDOR SALES
# ============================================================

vendor_sales = pd.read_sql(
    """
    SELECT *
    FROM analytics.vw_vendor_sales;
    """,
    engine
)

print("Vendors:", len(vendor_sales))

display(vendor_sales.head())


# ============================================================
# TOP VENDORS
# ============================================================

top_vendors = (
    vendor_sales
    .sort_values("revenue", ascending=False)
    .head(10)
)

display(top_vendors)


# ============================================================
# LOAD END INVENTORY
# ============================================================

end_inventory = pd.read_sql(
    """
    SELECT
        e.inventory_id,
        e.store_id,
        e.brand_id,
        p.description,
        p.size,
        e.on_hand,
        e.price,
        e.end_date
    FROM analytics.fact_end_inventory e
    LEFT JOIN analytics.dim_product p
        ON p.brand_id = e.brand_id;
    """,
    engine
)

print("Inventory rows:", len(end_inventory))

display(end_inventory.head())


# ============================================================
# INVENTORY ANALYSIS
# ============================================================

print("Total inventory units:")
print(end_inventory["on_hand"].sum())

print("\nZero inventory records:")
print(
    (end_inventory["on_hand"] == 0).sum()
)

print("\nLow inventory records <= 10:")
print(
    (end_inventory["on_hand"] <= 10).sum()
)


# ============================================================
#  LOW STOCK PRODUCTS
# ============================================================

low_stock = (
    end_inventory[
        end_inventory["on_hand"] <= 10
    ]
    .sort_values("on_hand")
)

display(low_stock.head(50))


# ============================================================
#  ZERO STOCK
# ============================================================

zero_stock = end_inventory[
    end_inventory["on_hand"] == 0
]

display(zero_stock.head(50))


# ============================================================
#  MONTHLY SALES USING PANDAS
# ============================================================

daily_kpi["month"] = (
    daily_kpi["sales_date"]
    .dt.to_period("M")
)

monthly_sales = (
    daily_kpi
    .groupby("month")
    .agg(
        revenue=("revenue", "sum"),
        units_sold=("units_sold", "sum"),
        sales_lines=("sales_lines", "sum")
    )
    .reset_index()
)

monthly_sales["month"] = (
    monthly_sales["month"]
    .astype(str)
)

display(monthly_sales)


# ============================================================
#  MONTH-OVER-MONTH GROWTH
# ============================================================

monthly_sales["previous_month_revenue"] = (
    monthly_sales["revenue"].shift(1)
)

monthly_sales["mom_growth_percent"] = (
    (
        monthly_sales["revenue"]
        - monthly_sales["previous_month_revenue"]
    )
    /
    monthly_sales["previous_month_revenue"]
    * 100
)

display(monthly_sales)


# ============================================================
#  CORRELATION
# ============================================================

correlation = daily_kpi[
    [
        "sales_lines",
        "units_sold",
        "revenue",
        "average_selling_price",
        "excise_tax"
    ]
].corr()

display(correlation)


# ============================================================
#  DAILY REVENUE DISTRIBUTION
# ============================================================

print(
    daily_kpi["revenue"].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)


# ============================================================
#  VISUALIZATION SETUP
# ============================================================

sns.set_theme(style="whitegrid")


# ============================================================
# DAILY REVENUE TREND
# ============================================================

plt.figure(figsize=(14, 6))

plt.plot(
    daily_kpi["sales_date"],
    daily_kpi["revenue"]
)

plt.title("Daily Revenue Trend")

plt.xlabel("Date")

plt.ylabel("Revenue")

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()


# ============================================================
#  MONTHLY REVENUE
# ============================================================

plt.figure(figsize=(10, 6))

plt.bar(
    monthly_sales["month"],
    monthly_sales["revenue"]
)

plt.title("Monthly Revenue")

plt.xlabel("Month")

plt.ylabel("Revenue")

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()


# ============================================================
#  TOP 10 PRODUCTS CHART
# ============================================================

plot_data = (
    product_sales
    .sort_values("revenue", ascending=False)
    .head(10)
    .sort_values("revenue")
)

plt.figure(figsize=(10, 6))

plt.barh(
    plot_data["description"],
    plot_data["revenue"]
)

plt.title("Top 10 Products by Revenue")

plt.xlabel("Revenue")

plt.ylabel("Product")

plt.tight_layout()

plt.show()


# ============================================================
#  TOP STORES CHART
# ============================================================

plot_data = (
    store_sales
    .sort_values("revenue", ascending=False)
    .head(10)
    .sort_values("revenue")
)

plt.figure(figsize=(10, 6))

plt.barh(
    plot_data["store_id"].astype(str),
    plot_data["revenue"]
)

plt.title("Top 10 Stores by Revenue")

plt.xlabel("Revenue")

plt.ylabel("Store")

plt.tight_layout()

plt.show()

# ============================================================
# CORRELATION HEATMAP
# ============================================================

plt.figure(figsize=(9, 7))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f"
)

plt.title("Sales KPI Correlation")

plt.tight_layout()

plt.show()


# ============================================================
# OUTLIER DETECTION USING IQR
# ============================================================

Q1 = daily_kpi["revenue"].quantile(0.25)

Q3 = daily_kpi["revenue"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR

upper_bound = Q3 + 1.5 * IQR

revenue_outliers = daily_kpi[
    (daily_kpi["revenue"] < lower_bound)
    |
    (daily_kpi["revenue"] > upper_bound)
]

print("Lower bound:", lower_bound)

print("Upper bound:", upper_bound)

print("Outlier days:", len(revenue_outliers))

display(revenue_outliers)


# ============================================================
# SALES PRICE DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

sns.histplot(
    product_sales["revenue"],
    bins=30,
    kde=True
)

plt.title("Product Revenue Distribution")

plt.xlabel("Revenue")

plt.ylabel("Number of Products")

plt.tight_layout()

plt.show()


# ============================================================
# BUSINESS INSIGHTS
# ============================================================

top_product = (
    product_sales
    .sort_values("revenue", ascending=False)
    .iloc[0]
)

top_store = (
    store_sales
    .sort_values("revenue", ascending=False)
    .iloc[0]
)

top_vendor = (
    vendor_sales
    .sort_values("revenue", ascending=False)
    .iloc[0]
)


print("==========================================")
print("AUTOMATED BUSINESS INSIGHTS")
print("==========================================")

print(
    f"Top Product: "
    f"{top_product['description']}"
)

print(
    f"Top Product Revenue: "
    f"${top_product['revenue']:,.2f}"
)

print(
    f"Top Store: "
    f"{top_store['store_id']}"
)

print(
    f"Top Store Revenue: "
    f"${top_store['revenue']:,.2f}"
)

print(
    f"Top Vendor: "
    f"{top_vendor['vendor_name']}"
)

print(
    f"Top Vendor Revenue: "
    f"${top_vendor['revenue']:,.2f}"
)



daily_kpi.to_csv(
    "daily_kpi.csv",
    index=False
)

monthly_sales.to_csv(
    "monthly_sales.csv",
    index=False
)

product_sales.to_csv(
    "product_sales_analysis.csv",
    index=False
)

store_sales.to_csv(
    "store_sales_analysis.csv",
    index=False
)

vendor_sales.to_csv(
    "vendor_sales_analysis.csv",
    index=False
)

abc_summary.to_csv(
    "abc_product_summary.csv",
    index=False
)

print("Analytical CSV files exported successfully.")


In [ ]:
# ============================================================
# PREPARE DAILY DATA
# ============================================================

daily = daily_kpi.copy()

daily["sales_date"] = pd.to_datetime(
    daily["sales_date"]
)

daily["year"] = daily["sales_date"].dt.year
daily["month"] = daily["sales_date"].dt.month
daily["month_name"] = daily["sales_date"].dt.month_name()
daily["quarter"] = daily["sales_date"].dt.quarter
daily["day_of_week"] = daily["sales_date"].dt.day_name()
daily["day_number"] = daily["sales_date"].dt.dayofweek

daily = daily.sort_values(
    "sales_date"
).reset_index(drop=True)

print("Daily dataset:", daily.shape)

display(daily.head())


# ============================================================
# DESCRIPTIVE STATISTICS
# ============================================================

numeric_cols = [
    "transaction_lines",
    "units_sold",
    "revenue",
    "avg_selling_price",
    "excise_tax"
]

print("========== DESCRIPTIVE STATISTICS ==========")

display(
    daily[numeric_cols].describe().T
)


# ============================================================
# SKEWNESS & KURTOSIS
# ============================================================

distribution_stats = pd.DataFrame({
    "skewness": daily[numeric_cols].skew(),
    "kurtosis": daily[numeric_cols].kurtosis()
})

print("========== DISTRIBUTION SHAPE ==========")

display(distribution_stats)


# ============================================================
# REVENUE DISTRIBUTION
# ============================================================

plt.figure(figsize=(12, 6))

sns.histplot(
    daily["revenue"],
    bins=30,
    kde=True
)

plt.title("Daily Revenue Distribution")
plt.xlabel("Revenue")
plt.ylabel("Number of Days")

plt.tight_layout()
plt.show()


# ============================================================
#  REVENUE BOXPLOT
# ============================================================

plt.figure(figsize=(12, 4))

sns.boxplot(
    x=daily["revenue"]
)

plt.title("Daily Revenue — Outlier Detection")
plt.xlabel("Revenue")

plt.tight_layout()
plt.show()


# ============================================================
#  IQR OUTLIER DETECTION
# ============================================================

def detect_iqr_outliers(df, column):

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    mask = (
        (df[column] < lower) |
        (df[column] > upper)
    )

    return {
        "Metric": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": int(mask.sum())
    }


outlier_results = []

for col in numeric_cols:

    outlier_results.append(
        detect_iqr_outliers(
            daily,
            col
        )
    )


outlier_df = pd.DataFrame(
    outlier_results
)

print("========== IQR OUTLIERS ==========")

display(outlier_df)


# ============================================================
# REVENUE OUTLIER DAYS
# ============================================================

Q1 = daily["revenue"].quantile(0.25)
Q3 = daily["revenue"].quantile(0.75)

IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR

revenue_outliers = daily[
    daily["revenue"] > upper_limit
].sort_values(
    "revenue",
    ascending=False
)

print("========== HIGH REVENUE OUTLIER DAYS ==========")

display(
    revenue_outliers[
        [
            "sales_date",
            "transaction_lines",
            "units_sold",
            "revenue",
            "avg_selling_price"
        ]
    ].head(20)
)


# ============================================================
#  Z-SCORE 
# ============================================================

revenue_mean = daily["revenue"].mean()
revenue_std = daily["revenue"].std()

daily["revenue_zscore"] = (
    (daily["revenue"] - revenue_mean)
    / revenue_std
)

zscore_outliers = daily[
    daily["revenue_zscore"].abs() > 3
].sort_values(
    "revenue_zscore",
    ascending=False
)

print("========== Z-SCORE ANOMALIES ==========")

display(
    zscore_outliers[
        [
            "sales_date",
            "revenue",
            "revenue_zscore"
        ]
    ]
)


# ============================================================
# MONTHLY ANALYSIS
# ============================================================

monthly = (
    daily
    .groupby(
        ["year", "month", "month_name"],
        as_index=False
    )
    .agg(
        revenue=("revenue", "sum"),
        units_sold=("units_sold", "sum"),
        transaction_lines=("transaction_lines", "sum"),
        avg_selling_price=("avg_selling_price", "mean"),
        excise_tax=("excise_tax", "sum")
    )
)

monthly = monthly.sort_values(
    ["year", "month"]
).reset_index(drop=True)

print("========== MONTHLY PERFORMANCE ==========")

display(monthly)


# ============================================================
#  MONTHLY REVENUE
# ============================================================

plt.figure(figsize=(14, 6))

sns.barplot(
    data=monthly,
    x="month_name",
    y="revenue"
)

plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
# QUARTERLY ANALYSIS
# ============================================================

quarterly = (
    daily
    .groupby(
        ["year", "quarter"],
        as_index=False
    )
    .agg(
        revenue=("revenue", "sum"),
        units_sold=("units_sold", "sum"),
        transaction_lines=("transaction_lines", "sum"),
        avg_selling_price=("avg_selling_price", "mean")
    )
)

print("========== QUARTERLY PERFORMANCE ==========")

display(quarterly)


# ============================================================
# DAY OF WEEK
# ============================================================

weekday = (
    daily
    .groupby(
        ["day_number", "day_of_week"],
        as_index=False
    )
    .agg(
        revenue=("revenue", "mean"),
        units_sold=("units_sold", "mean"),
        transaction_lines=("transaction_lines", "mean")
    )
    .sort_values("day_number")
)

print("========== DAY-OF-WEEK ANALYSIS ==========")

display(weekday)


# ============================================================
#  DAY OF WEEK CHART
# ============================================================

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

plt.figure(figsize=(12, 6))

sns.barplot(
    data=weekday,
    x="day_of_week",
    y="revenue",
    order=weekday_order
)

plt.title("Average Revenue by Day of Week")
plt.xlabel("Day")
plt.ylabel("Average Revenue")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
# DAILY REVENUE TREND
# ============================================================

plt.figure(figsize=(16, 6))

plt.plot(
    daily["sales_date"],
    daily["revenue"]
)

plt.title("Daily Revenue Trend")
plt.xlabel("Date")
plt.ylabel("Revenue")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
# 7-DAY MOVING AVERAGE
# ============================================================

daily["revenue_7d_avg"] = (
    daily["revenue"]
    .rolling(7)
    .mean()
)

plt.figure(figsize=(16, 6))

plt.plot(
    daily["sales_date"],
    daily["revenue"],
    alpha=0.4,
    label="Daily Revenue"
)

plt.plot(
    daily["sales_date"],
    daily["revenue_7d_avg"],
    linewidth=2,
    label="7-Day Moving Average"
)

plt.title("Daily Revenue + 7-Day Moving Average")
plt.xlabel("Date")
plt.ylabel("Revenue")

plt.legend()

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
#  DAILY VOLATILITY
# ============================================================

daily["revenue_change"] = (
    daily["revenue"].diff()
)

daily["revenue_growth_pct"] = (
    daily["revenue"]
    .pct_change()
    * 100
)

print("========== DAILY REVENUE CHANGE ==========")

display(
    daily[
        [
            "sales_date",
            "revenue",
            "revenue_change",
            "revenue_growth_pct"
        ]
    ].head(15)
)


# ============================================================
# MOST VOLATILE DAYS
# ============================================================

most_volatile = (
    daily
    .dropna(
        subset=["revenue_growth_pct"]
    )
    .sort_values(
        "revenue_growth_pct",
        key=lambda x: x.abs(),
        ascending=False
    )
)

print("========== MOST VOLATILE DAYS ==========")

display(
    most_volatile[
        [
            "sales_date",
            "revenue",
            "revenue_growth_pct"
        ]
    ].head(20)
)


# ============================================================
# CORRELATION
# ============================================================

corr_matrix = daily[
    numeric_cols
].corr()

print("========== CORRELATION MATRIX ==========")

display(corr_matrix)


# ============================================================
#  CORRELATION HEATMAP
# ============================================================

plt.figure(figsize=(10, 7))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Daily KPI Correlation")

plt.tight_layout()
plt.show()


# ============================================================
# TRANSACTION LINES VS REVENUE
# ============================================================

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=daily,
    x="transaction_lines",
    y="revenue"
)

plt.title("Transaction Lines vs Revenue")
plt.xlabel("Transaction Lines")
plt.ylabel("Revenue")

plt.tight_layout()
plt.show()


# ============================================================
# UNITS VS REVENUE
# ============================================================

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=daily,
    x="units_sold",
    y="revenue"
)

plt.title("Units Sold vs Revenue")
plt.xlabel("Units Sold")
plt.ylabel("Revenue")

plt.tight_layout()
plt.show()


# ============================================================
#LOAD PRODUCT DATA
# ============================================================

product_sales = pd.read_sql(
    """
    SELECT *
    FROM analytics.vw_product_sales
    ORDER BY revenue DESC;
    """,
    engine
)

print("Product dataset:", product_sales.shape)

display(product_sales.head(10))


# ============================================================
# PRODUCT REVENUE CONCENTRATION
# ============================================================

total_product_revenue = (
    product_sales["revenue"].sum()
)

product_sales["revenue_share_pct"] = (
    product_sales["revenue"]
    / total_product_revenue
    * 100
)

product_sales["cumulative_revenue_pct"] = (
    product_sales["revenue_share_pct"]
    .cumsum()
)

print("========== TOP PRODUCTS ==========")

display(
    product_sales[
        [
            "brand_id",
            "description",
            "revenue",
            "revenue_share_pct",
            "cumulative_revenue_pct"
        ]
    ].head(20)
)


# ============================================================
# TOP 20 PRODUCTS
# ============================================================

plt.figure(figsize=(14, 7))

sns.barplot(
    data=product_sales.head(20),
    x="revenue",
    y="description"
)

plt.title("Top 20 Products by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product")

plt.tight_layout()
plt.show()


# ============================================================
# 80% REVENUE ANALYSIS
# ============================================================

products_for_80pct = (
    product_sales[
        product_sales["cumulative_revenue_pct"] <= 80
    ]
)

print(
    "Products contributing to first 80% of revenue:",
    len(products_for_80pct)
)

print(
    "Total products:",
    len(product_sales)
)

print(
    "Percentage of products:",
    round(
        len(products_for_80pct)
        / len(product_sales)
        * 100,
        2
    ),
    "%"
)


# ============================================================
# STORE PERFORMANCE
# ============================================================

store_sales = pd.read_sql(
    """
    SELECT *
    FROM analytics.vw_store_sales
    ORDER BY revenue DESC;
    """,
    engine
)

print("Store dataset:", store_sales.shape)

display(store_sales.head(10))


# ============================================================
# STORE REVENUE DISTRIBUTION
# ============================================================

plt.figure(figsize=(12, 6))

sns.histplot(
    store_sales["revenue"],
    bins=20,
    kde=True
)

plt.title("Store Revenue Distribution")
plt.xlabel("Revenue")
plt.ylabel("Number of Stores")

plt.tight_layout()
plt.show()


# ============================================================
# TOP / BOTTOM STORES
# ============================================================

print("========== TOP 10 STORES ==========")

display(
    store_sales.head(10)
)

print("========== BOTTOM 10 STORES ==========")

display(
    store_sales.tail(10)
)


# ============================================================
# STORE STATISTICS
# ============================================================

display(
    store_sales["revenue"]
    .describe()
    .to_frame("revenue")
)


# ============================================================
# LOAD ENDING INVENTORY
# ============================================================

inventory = pd.read_sql(
    """
    SELECT
        inventory_id,
        store_id,
        brand_id,
        on_hand,
        price,
        end_date
    FROM analytics.fact_end_inventory;
    """,
    engine
)

print("Inventory rows:", len(inventory))

display(inventory.head())


# ============================================================
# INVENTORY VALUE
# ============================================================

inventory["inventory_value"] = (
    inventory["on_hand"]
    * inventory["price"]
)

print(
    "Total inventory value:",
    round(
        inventory["inventory_value"].sum(),
        2
    )
)

print(
    "Average inventory value:",
    round(
        inventory["inventory_value"].mean(),
        2
    )
)


# ============================================================
# INVENTORY DISTRIBUTION
# ============================================================

plt.figure(figsize=(12, 6))

sns.histplot(
    inventory["on_hand"],
    bins=40
)

plt.title("Ending Inventory Distribution")
plt.xlabel("Units on Hand")
plt.ylabel("Inventory Records")

plt.tight_layout()
plt.show()


# ============================================================
# LOW / ZERO INVENTORY
# ============================================================

low_inventory = inventory[
    inventory["on_hand"] <= 5
]

zero_inventory = inventory[
    inventory["on_hand"] == 0
]

print(
    "Records with <= 5 units:",
    len(low_inventory)
)

print(
    "Records with zero units:",
    len(zero_inventory)
)


# ============================================================
#  INVENTORY BY STORE
# ============================================================

inventory_by_store = (
    inventory
    .groupby(
        "store_id",
        as_index=False
    )
    .agg(
        inventory_units=("on_hand", "sum"),
        inventory_value=("inventory_value", "sum")
    )
    .sort_values(
        "inventory_value",
        ascending=False
    )
)

print("========== INVENTORY BY STORE ==========")

display(
    inventory_by_store.head(20)
)


# Compare average daily revenue:
#
# First half: January–June
# Second half: July–December

first_half = daily[
    daily["month"] <= 6
]["revenue"].dropna()

second_half = daily[
    daily["month"] >= 7
]["revenue"].dropna()

n1 = len(first_half)
n2 = len(second_half)

mean1 = first_half.mean()
mean2 = second_half.mean()

var1 = first_half.var(ddof=1)
var2 = second_half.var(ddof=1)

standard_error = np.sqrt(
    (var1 / n1) +
    (var2 / n2)
)

t_statistic = (
    (mean1 - mean2)
    / standard_error
)

# Welch-Satterthwaite degrees of freedom
numerator = (
    (var1 / n1) +
    (var2 / n2)
) ** 2

denominator = (
    ((var1 / n1) ** 2) / (n1 - 1)
    +
    ((var2 / n2) ** 2) / (n2 - 1)
)

degrees_freedom = (
    numerator / denominator
)

print("========== HALF-YEAR COMPARISON ==========")

print(
    "First-half mean revenue:",
    round(mean1, 2)
)

print(
    "Second-half mean revenue:",
    round(mean2, 2)
)

print(
    "T-statistic:",
    round(t_statistic, 4)
)

print(
    "Approx. degrees of freedom:",
    round(degrees_freedom, 2)
)
# ============================================================

mean_revenue = daily["revenue"].mean()

std_revenue = daily["revenue"].std(
    ddof=1
)

n = len(daily)

standard_error = (
    std_revenue
    / np.sqrt(n)
)

margin_error = (
    1.96
    * standard_error
)

lower_ci = (
    mean_revenue
    - margin_error
)

upper_ci = (
    mean_revenue
    + margin_error
)

print("========== 95% CONFIDENCE INTERVAL ==========")

print(
    "Mean daily revenue:",
    round(mean_revenue, 2)
)

print(
    "Lower bound:",
    round(lower_ci, 2)
)

print(
    "Upper bound:",
    round(upper_ci, 2)
)


# ============================================================
# AUTOMATED BUSINESS INSIGHTS
# ============================================================

best_day = daily.loc[
    daily["revenue"].idxmax()
]

worst_day = daily.loc[
    daily["revenue"].idxmin()
]

best_month = monthly.loc[
    monthly["revenue"].idxmax()
]

best_store = store_sales.iloc[0]

top_product = product_sales.iloc[0]


print("\n")
print("=" * 65)
print("PHASE 10 — BUSINESS INSIGHTS")
print("=" * 65)

print(
    f"\nHighest revenue day: "
    f"{best_day['sales_date'].date()} "
    f"→ ₹{best_day['revenue']:,.2f}"
)

print(
    f"Lowest revenue day: "
    f"{worst_day['sales_date'].date()} "
    f"→ ₹{worst_day['revenue']:,.2f}"
)

print(
    f"\nBest month: "
    f"{best_month['month_name']} "
    f"→ ₹{best_month['revenue']:,.2f}"
)

print(
    f"\nTop store: "
    f"Store {best_store['store_id']} "
    f"→ ₹{best_store['revenue']:,.2f}"
)

print(
    f"\nTop product: "
    f"{top_product['description']} "
    f"→ ₹{top_product['revenue']:,.2f}"
)

print(
    f"\nAverage daily revenue: "
    f"₹{daily['revenue'].mean():,.2f}"
)

print(
    f"Revenue standard deviation: "
    f"₹{daily['revenue'].std():,.2f}"
)

print(
    f"\nTotal inventory value: "
    f"₹{inventory['inventory_value'].sum():,.2f}"
)

print(
    f"Zero-inventory records: "
    f"{len(zero_inventory):,}"
)


# ============================================================
# EXPORT RESULTS
# ============================================================

monthly.to_csv(
    "phase10_monthly_analysis.csv",
    index=False
)

weekday.to_csv(
    "phase10_weekday_analysis.csv",
    index=False
)

outlier_df.to_csv(
    "phase10_outlier_analysis.csv",
    index=False
)

store_sales.to_csv(
    "phase10_store_analysis.csv",
    index=False
)

product_sales.head(1000).to_csv(
    "phase10_product_analysis_top1000.csv",
    index=False
)

inventory_by_store.to_csv(
    "phase10_inventory_by_store.csv",
    index=False
)

print("\nEDA result files exported successfully.")




In [ ]:
# ============================================================
#  VISUALIZATION SETTINGS
# ============================================================

sns.set_theme(
    style="whitegrid"
)

plt.rcParams["figure.figsize"] = (12, 6)

print("Visualization environment ready.")


# ============================================================
# PREPARE DAILY DATA
# ============================================================

daily_viz = daily.copy()

daily_viz["sales_date"] = pd.to_datetime(
    daily_viz["sales_date"]
)

daily_viz = daily_viz.sort_values(
    "sales_date"
).reset_index(drop=True)

daily_viz["month"] = (
    daily_viz["sales_date"]
    .dt.month
)

daily_viz["month_name"] = (
    daily_viz["sales_date"]
    .dt.month_name()
)

daily_viz["day_of_week"] = (
    daily_viz["sales_date"]
    .dt.day_name()
)

daily_viz["day_number"] = (
    daily_viz["sales_date"]
    .dt.dayofweek
)

print("Daily visualization dataset:", daily_viz.shape)


# ============================================================
# KPI SUMMARY
# ============================================================

total_revenue = daily_viz["revenue"].sum()

total_units = daily_viz["units_sold"].sum()

total_transactions = (
    daily_viz["transaction_lines"].sum()
)

average_daily_revenue = (
    daily_viz["revenue"].mean()
)

average_price = (
    daily_viz["avg_selling_price"].mean()
)

total_excise_tax = (
    daily_viz["excise_tax"].sum()
)

print("=" * 60)
print("KEY BUSINESS KPIs")
print("=" * 60)

print(
    f"Total Revenue        : ₹{total_revenue:,.2f}"
)

print(
    f"Total Units Sold     : {total_units:,}"
)

print(
    f"Transaction Lines    : {total_transactions:,}"
)

print(
    f"Average Daily Revenue: ₹{average_daily_revenue:,.2f}"
)

print(
    f"Average Selling Price: ₹{average_price:,.2f}"
)

print(
    f"Total Excise Tax     : ₹{total_excise_tax:,.2f}"
)


# ============================================================
# DAILY REVENUE TREND
# ============================================================

plt.figure(figsize=(16, 6))

plt.plot(
    daily_viz["sales_date"],
    daily_viz["revenue"],
    linewidth=1.5
)

plt.title(
    "Daily Revenue Trend",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Date")
plt.ylabel("Revenue")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
#  DAILY REVENUE + MOVING AVERAGE
# ============================================================

daily_viz["revenue_7d_avg"] = (
    daily_viz["revenue"]
    .rolling(7)
    .mean()
)

plt.figure(figsize=(16, 6))

plt.plot(
    daily_viz["sales_date"],
    daily_viz["revenue"],
    alpha=0.35,
    label="Daily Revenue"
)

plt.plot(
    daily_viz["sales_date"],
    daily_viz["revenue_7d_avg"],
    linewidth=2.5,
    label="7-Day Moving Average"
)

plt.title(
    "Revenue Trend with 7-Day Moving Average",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Date")
plt.ylabel("Revenue")

plt.legend()

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
#  MONTHLY REVENUE
# ============================================================

monthly_viz = (
    daily_viz
    .groupby(
        ["month", "month_name"],
        as_index=False
    )
    .agg(
        revenue=("revenue", "sum"),
        units_sold=("units_sold", "sum"),
        transactions=("transaction_lines", "sum")
    )
    .sort_values("month")
)

print("========== MONTHLY REVENUE ==========")

display(monthly_viz)


# ============================================================
#  MONTHLY REVENUE BAR CHART
# ============================================================

plt.figure(figsize=(14, 6))

sns.barplot(
    data=monthly_viz,
    x="month_name",
    y="revenue"
)

plt.title(
    "Monthly Revenue",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Month")
plt.ylabel("Revenue")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
#  MONTHLY REVENUE + UNITS
# ============================================================

fig, ax1 = plt.subplots(
    figsize=(14, 6)
)

ax1.plot(
    monthly_viz["month_name"],
    monthly_viz["revenue"],
    marker="o",
    linewidth=2,
    label="Revenue"
)

ax1.set_xlabel("Month")
ax1.set_ylabel("Revenue")

ax2 = ax1.twinx()

ax2.plot(
    monthly_viz["month_name"],
    monthly_viz["units_sold"],
    marker="o",
    linewidth=2,
    label="Units Sold"
)

ax2.set_ylabel("Units Sold")

plt.title(
    "Monthly Revenue vs Units Sold",
    fontsize=16,
    fontweight="bold"
)

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
#  DAY-OF-WEEK PERFORMANCE
# ============================================================

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_viz = (
    daily_viz
    .groupby(
        ["day_number", "day_of_week"],
        as_index=False
    )
    .agg(
        avg_revenue=("revenue", "mean"),
        avg_units=("units_sold", "mean"),
        avg_transactions=("transaction_lines", "mean")
    )
    .sort_values("day_number")
)

print("========== DAY OF WEEK ==========")

display(weekday_viz)


# ============================================================
#  DAY-OF-WEEK REVENUE
# ============================================================

plt.figure(figsize=(12, 6))

sns.barplot(
    data=weekday_viz,
    x="day_of_week",
    y="avg_revenue",
    order=weekday_order
)

plt.title(
    "Average Revenue by Day of Week",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Day")
plt.ylabel("Average Revenue")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
#  REVENUE HEATMAP
# ============================================================

daily_viz["weekday"] = (
    daily_viz["sales_date"]
    .dt.day_name()
)

heatmap_data = (
    daily_viz
    .pivot_table(
        index="month_name",
        columns="weekday",
        values="revenue",
        aggfunc="mean"
    )
)

heatmap_data = heatmap_data.reindex(
    columns=weekday_order
)

month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

heatmap_data = heatmap_data.reindex(
    month_order
)

plt.figure(figsize=(14, 8))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".0f",
    cmap="YlOrRd"
)

plt.title(
    "Average Daily Revenue by Month and Day",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Day of Week")
plt.ylabel("Month")

plt.tight_layout()
plt.show()


# ============================================================
#  TOP PRODUCTS
# ============================================================

product_viz = product_sales.copy()

product_viz = product_viz.sort_values(
    "revenue",
    ascending=False
)

top_products = product_viz.head(15)

print("========== TOP 15 PRODUCTS ==========")

display(
    top_products[
        [
            "brand_id",
            "description",
            "units_sold",
            "revenue"
        ]
    ]
)


# ============================================================
#  TOP PRODUCTS CHART
# ============================================================

plt.figure(figsize=(14, 8))

sns.barplot(
    data=top_products,
    x="revenue",
    y="description"
)

plt.title(
    "Top 15 Products by Revenue",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Revenue")
plt.ylabel("Product")

plt.tight_layout()
plt.show()


# ============================================================
#  TOP PRODUCTS BY UNITS
# ============================================================

top_units = (
    product_viz
    .sort_values(
        "units_sold",
        ascending=False
    )
    .head(15)
)

plt.figure(figsize=(14, 8))

sns.barplot(
    data=top_units,
    x="units_sold",
    y="description"
)

plt.title(
    "Top 15 Products by Units Sold",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Units Sold")
plt.ylabel("Product")

plt.tight_layout()
plt.show()


# ============================================================
# PRODUCT REVENUE CONCENTRATION
# ============================================================

product_viz["revenue_share"] = (
    product_viz["revenue"]
    / product_viz["revenue"].sum()
    * 100
)

product_viz["cumulative_revenue"] = (
    product_viz["revenue_share"]
    .cumsum()
)

plt.figure(figsize=(14, 6))

plt.plot(
    range(1, len(product_viz) + 1),
    product_viz["cumulative_revenue"]
)

plt.axhline(
    80,
    linestyle="--",
    label="80% Revenue"
)

plt.title(
    "Product Revenue Concentration",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Products Ranked by Revenue")
plt.ylabel("Cumulative Revenue %")

plt.legend()

plt.tight_layout()
plt.show()


# ============================================================
# TOP STORES
# ============================================================

store_viz = store_sales.copy()

store_viz = store_viz.sort_values(
    "revenue",
    ascending=False
)

print("========== TOP 15 STORES ==========")

display(
    store_viz.head(15)
)


# ============================================================
#  STORE REVENUE CHART
# ============================================================

plt.figure(figsize=(14, 8))

sns.barplot(
    data=store_viz.head(15),
    x="revenue",
    y="store_id"
)

plt.title(
    "Top 15 Stores by Revenue",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Revenue")
plt.ylabel("Store")

plt.tight_layout()
plt.show()


# ============================================================
# STORE REVENUE DISTRIBUTION
# ============================================================

plt.figure(figsize=(12, 6))

sns.histplot(
    store_viz["revenue"],
    bins=20,
    kde=True
)

plt.title(
    "Store Revenue Distribution",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Revenue")
plt.ylabel("Number of Stores")

plt.tight_layout()
plt.show()


# ============================================================
#  STORE PERFORMANCE BOXPLOT
# ============================================================

plt.figure(figsize=(12, 5))

sns.boxplot(
    x=store_viz["revenue"]
)

plt.title(
    "Store Revenue Distribution & Outliers",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Revenue")

plt.tight_layout()
plt.show()


# ============================================================
# INVENTORY ANALYSIS
# ============================================================

inventory_viz = inventory.copy()

inventory_viz["inventory_value"] = (
    inventory_viz["on_hand"]
    * inventory_viz["price"]
)

print("========== INVENTORY SUMMARY ==========")

display(
    inventory_viz[
        [
            "on_hand",
            "price",
            "inventory_value"
        ]
    ].describe()
)


# ============================================================
#  INVENTORY VALUE BY STORE
# ============================================================

inventory_store = (
    inventory_viz
    .groupby(
        "store_id",
        as_index=False
    )
    .agg(
        inventory_units=("on_hand", "sum"),
        inventory_value=("inventory_value", "sum")
    )
    .sort_values(
        "inventory_value",
        ascending=False
    )
)

print("========== TOP STORES BY INVENTORY VALUE ==========")

display(
    inventory_store.head(15)
)


# ============================================================
# INVENTORY VALUE CHART
# ============================================================

plt.figure(figsize=(14, 8))

sns.barplot(
    data=inventory_store.head(15),
    x="inventory_value",
    y="store_id"
)

plt.title(
    "Top 15 Stores by Inventory Value",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Inventory Value")
plt.ylabel("Store")

plt.tight_layout()
plt.show()


# ============================================================
#  LOW INVENTORY DISTRIBUTION
# ============================================================

low_inventory = inventory_viz[
    inventory_viz["on_hand"] <= 5
]

inventory_status = pd.DataFrame({
    "status": [
        "Zero Stock",
        "1–5 Units",
        "More Than 5"
    ],
    "records": [
        (inventory_viz["on_hand"] == 0).sum(),
        (
            (inventory_viz["on_hand"] >= 1) &
            (inventory_viz["on_hand"] <= 5)
        ).sum(),
        (inventory_viz["on_hand"] > 5).sum()
    ]
})

print("========== INVENTORY STATUS ==========")

display(inventory_status)


# ============================================================
#  INVENTORY STATUS CHART
# ============================================================

plt.figure(figsize=(10, 6))

sns.barplot(
    data=inventory_status,
    x="status",
    y="records"
)

plt.title(
    "Inventory Stock Status",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Inventory Status")
plt.ylabel("Records")

plt.tight_layout()
plt.show()


# ============================================================
#  KPI CORRELATION HEATMAP
# ============================================================

correlation_cols = [
    "transaction_lines",
    "units_sold",
    "revenue",
    "avg_selling_price",
    "excise_tax"
]

correlation = daily_viz[
    correlation_cols
].corr()

plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title(
    "KPI Correlation Matrix",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()


# ============================================================
#  REVENUE VS TRANSACTIONS
# ============================================================

plt.figure(figsize=(10, 6))

sns.regplot(
    data=daily_viz,
    x="transaction_lines",
    y="revenue",
    scatter_kws={"alpha": 0.5}
)

plt.title(
    "Revenue vs Transaction Lines",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Transaction Lines")
plt.ylabel("Revenue")

plt.tight_layout()
plt.show()


# ============================================================
# REVENUE VS UNITS
# ============================================================

plt.figure(figsize=(10, 6))

sns.regplot(
    data=daily_viz,
    x="units_sold",
    y="revenue",
    scatter_kws={"alpha": 0.5}
)

plt.title(
    "Revenue vs Units Sold",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Units Sold")
plt.ylabel("Revenue")

plt.tight_layout()
plt.show()


# ============================================================
# PRICE DISTRIBUTION
# ============================================================

plt.figure(figsize=(12, 6))

sns.histplot(
    daily_viz["avg_selling_price"],
    bins=30,
    kde=True
)

plt.title(
    "Average Selling Price Distribution",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Average Selling Price")
plt.ylabel("Number of Days")

plt.tight_layout()
plt.show()


# ============================================================
#  HIGH / LOW REVENUE DAYS
# ============================================================

highest_days = (
    daily_viz
    .nlargest(
        10,
        "revenue"
    )
)

lowest_days = (
    daily_viz
    .nsmallest(
        10,
        "revenue"
    )
)

print("========== 10 HIGHEST REVENUE DAYS ==========")

display(
    highest_days[
        [
            "sales_date",
            "revenue",
            "units_sold",
            "transaction_lines"
        ]
    ]
)

print("========== 10 LOWEST REVENUE DAYS ==========")

display(
    lowest_days[
        [
            "sales_date",
            "revenue",
            "units_sold",
            "transaction_lines"
        ]
    ]
)


# ============================================================
#  BUSINESS STORY SUMMARY
# ============================================================

best_day = daily_viz.loc[
    daily_viz["revenue"].idxmax()
]

worst_day = daily_viz.loc[
    daily_viz["revenue"].idxmin()
]

best_month = monthly_viz.loc[
    monthly_viz["revenue"].idxmax()
]

best_weekday = weekday_viz.loc[
    weekday_viz["avg_revenue"].idxmax()
]

best_store = store_viz.iloc[0]

best_product = product_viz.iloc[0]


print("\n")
print("=" * 70)
print("DATA STORY — KEY FINDINGS")
print("=" * 70)

print(
    f"""
1. Overall Revenue
   Total revenue: ₹{total_revenue:,.2f}

2. Strongest Sales Day
   {best_day['sales_date'].date()}
   Revenue: ₹{best_day['revenue']:,.2f}

3. Weakest Sales Day
   {worst_day['sales_date'].date()}
   Revenue: ₹{worst_day['revenue']:,.2f}

4. Strongest Month
   {best_month['month_name']}
   Revenue: ₹{best_month['revenue']:,.2f}

5. Strongest Weekday
   {best_weekday['day_of_week']}
   Average revenue: ₹{best_weekday['avg_revenue']:,.2f}

6. Top Store
   Store {best_store['store_id']}
   Revenue: ₹{best_store['revenue']:,.2f}

7. Top Product
   {best_product['description']}
   Revenue: ₹{best_product['revenue']:,.2f}

8. Inventory
   Total inventory value:
   ₹{inventory_viz['inventory_value'].sum():,.2f}
"""
)


# ============================================================
# EXPORT VISUALIZATION DATA
# ============================================================

monthly_viz.to_csv(
    "phase11_monthly_visualization.csv",
    index=False
)

weekday_viz.to_csv(
    "phase11_weekday_visualization.csv",
    index=False
)

product_viz.head(1000).to_csv(
    "phase11_product_visualization.csv",
    index=False
)

store_viz.to_csv(
    "phase11_store_visualization.csv",
    index=False
)

inventory_store.to_csv(
    "phase11_inventory_visualization.csv",
    index=False
)

heatmap_data.to_csv(
    "phase11_revenue_heatmap.csv"
)

print("\nVisualization datasets exported successfully.")


In [ ]:
# ============================================================
# LOAD BUSINESS DATA FROM POSTGRESQL
# ============================================================

# Daily sales
daily = pd.read_sql("""
    SELECT *
    FROM analytics.vw_daily_kpi
    ORDER BY sales_date
""", engine)

# Product sales
product_sales = pd.read_sql("""
    SELECT *
    FROM analytics.vw_product_sales
    ORDER BY revenue DESC
""", engine)

# Store sales
store_sales = pd.read_sql("""
    SELECT *
    FROM analytics.vw_store_sales
    ORDER BY revenue DESC
""", engine)

# Vendor sales
vendor_sales = pd.read_sql("""
    SELECT *
    FROM analytics.vw_vendor_sales
    ORDER BY revenue DESC
""", engine)

# Inventory position
inventory = pd.read_sql("""
    SELECT
        e.store_id,
        e.brand_id,
        p.description,
        p.size,
        p.classification,
        e.on_hand,
        e.price,
        e.end_date,
        e.on_hand * e.price AS inventory_value
    FROM analytics.fact_end_inventory e
    LEFT JOIN analytics.dim_product p
        ON e.brand_id = p.brand_id
""", engine)

# Purchase summary
purchase_summary = pd.read_sql("""
    SELECT
        brand_id,
        vendor_number,
        SUM(quantity) AS purchase_units,
        SUM(dollars) AS purchase_cost,
        CASE
            WHEN SUM(quantity) > 0
            THEN SUM(dollars) / SUM(quantity)
            ELSE NULL
        END AS weighted_purchase_cost
    FROM analytics.fact_purchases
    GROUP BY brand_id, vendor_number
""", engine)

print("Daily rows:", len(daily))
print("Products:", len(product_sales))
print("Stores:", len(store_sales))
print("Vendors:", len(vendor_sales))
print("Inventory rows:", len(inventory))
print("Purchase summaries:", len(purchase_summary))


# ============================================================
#  OVERALL BUSINESS KPIs
# ============================================================

print("\n" + "=" * 70)
print("2. OVERALL BUSINESS KPIs")
print("=" * 70)

total_revenue = product_sales["revenue"].sum()
total_units = product_sales["units_sold"].sum()
total_transactions = daily["transaction_lines"].sum()
total_inventory_value = inventory["inventory_value"].sum()
total_purchase_cost = purchase_summary["purchase_cost"].sum()

avg_selling_price = total_revenue / total_units if total_units else np.nan

print(f"Total Revenue          : ${total_revenue:,.2f}")
print(f"Total Units Sold       : {total_units:,.0f}")
print(f"Transaction Lines      : {total_transactions:,.0f}")
print(f"Average Selling Price  : ${avg_selling_price:,.2f}")
print(f"Inventory Value        : ${total_inventory_value:,.2f}")
print(f"Purchase Cost          : ${total_purchase_cost:,.2f}")


# ============================================================
#  PRODUCT PERFORMANCE
# ============================================================

print("\n" + "=" * 70)
print("3. PRODUCT PERFORMANCE")
print("=" * 70)

product = product_sales.copy()

product["revenue_share"] = (
    product["revenue"] / product["revenue"].sum() * 100
)

product["unit_share"] = (
    product["units_sold"] / product["units_sold"].sum() * 100
)

product["revenue_rank"] = (
    product["revenue"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

print("\nTOP 10 PRODUCTS BY REVENUE")
print(
    product[
        [
            "brand_id",
            "description",
            "revenue",
            "units_sold",
            "avg_selling_price",
            "revenue_share"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\nTOP 10 PRODUCTS BY UNITS")
print(
    product.sort_values("units_sold", ascending=False)[
        [
            "brand_id",
            "description",
            "units_sold",
            "revenue",
            "avg_selling_price"
        ]
    ]
    .head(10)
    .to_string(index=False)
)


# ============================================================
#  ABC PRODUCT CLASSIFICATION
# ============================================================

print("\n" + "=" * 70)
print("4. ABC PRODUCT CLASSIFICATION")
print("=" * 70)

abc = product.sort_values("revenue", ascending=False).copy()

abc["cumulative_revenue"] = abc["revenue"].cumsum()

abc["cumulative_percent"] = (
    abc["cumulative_revenue"]
    / abc["revenue"].sum()
    * 100
)

abc["ABC_Class"] = np.select(
    [
        abc["cumulative_percent"] <= 80,
        abc["cumulative_percent"] <= 95
    ],
    [
        "A",
        "B"
    ],
    default="C"
)

abc_summary = (
    abc.groupby("ABC_Class")
    .agg(
        products=("brand_id", "count"),
        revenue=("revenue", "sum"),
        units=("units_sold", "sum")
    )
    .reset_index()
)

abc_summary["revenue_share"] = (
    abc_summary["revenue"]
    / abc_summary["revenue"].sum()
    * 100
)

print(abc_summary.to_string(index=False))

print("\nInterpretation:")
print("A = highest-value products")
print("B = medium-value products")
print("C = lower-value / long-tail products")


# ============================================================
# PRODUCT COST & ESTIMATED MARGIN
# ============================================================

print("\n" + "=" * 70)
print("5. ESTIMATED PRODUCT MARGIN")
print("=" * 70)

product_cost = pd.read_sql("""
    SELECT
        brand_id,
        SUM(quantity) AS purchased_units,
        SUM(dollars) AS purchase_cost,
        CASE
            WHEN SUM(quantity) > 0
            THEN SUM(dollars) / SUM(quantity)
            ELSE NULL
        END AS weighted_avg_cost
    FROM analytics.fact_purchases
    GROUP BY brand_id
""", engine)

margin = product_sales.merge(
    product_cost,
    on="brand_id",
    how="left"
)

margin["estimated_cogs"] = (
    margin["units_sold"] *
    margin["weighted_avg_cost"]
)

margin["estimated_gross_profit"] = (
    margin["revenue"] -
    margin["estimated_cogs"]
)

margin["estimated_margin_percent"] = np.where(
    margin["revenue"] != 0,
    margin["estimated_gross_profit"]
    / margin["revenue"] * 100,
    np.nan
)

print("\nTOP 10 PRODUCTS BY ESTIMATED GROSS PROFIT")

print(
    margin.sort_values(
        "estimated_gross_profit",
        ascending=False
    )[
        [
            "brand_id",
            "description",
            "revenue",
            "units_sold",
            "weighted_avg_cost",
            "estimated_gross_profit",
            "estimated_margin_percent"
        ]
    ]
    .head(10)
    .to_string(index=False)
)


# ============================================================
#  LOW-MARGIN PRODUCTS
# ============================================================

print("\n" + "=" * 70)
print("6. LOW-MARGIN PRODUCTS")
print("=" * 70)

low_margin = (
    margin[
        margin["estimated_margin_percent"].notna()
    ]
    .sort_values("estimated_margin_percent")
)

print(
    low_margin[
        [
            "brand_id",
            "description",
            "revenue",
            "weighted_avg_cost",
            "estimated_margin_percent"
        ]
    ]
    .head(15)
    .to_string(index=False)
)


# ============================================================
#  STORE PERFORMANCE
# ============================================================

print("\n" + "=" * 70)
print("7. STORE PERFORMANCE")
print("=" * 70)

stores = store_sales.copy()

stores["revenue_share"] = (
    stores["revenue"]
    / stores["revenue"].sum()
    * 100
)

stores["revenue_rank"] = (
    stores["revenue"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

print("\nTOP 10 STORES")

print(
    stores[
        [
            "store_id",
            "city",
            "revenue",
            "units_sold",
            "avg_selling_price",
            "revenue_share"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\nBOTTOM 10 STORES")

print(
    stores.sort_values("revenue")[
        [
            "store_id",
            "city",
            "revenue",
            "units_sold",
            "avg_selling_price"
        ]
    ]
    .head(10)
    .to_string(index=False)
)


# ============================================================
#  VENDOR PERFORMANCE
# ============================================================

print("\n" + "=" * 70)
print("8. VENDOR PERFORMANCE")
print("=" * 70)

vendors = vendor_sales.copy()

vendors["revenue_share"] = (
    vendors["revenue"]
    / vendors["revenue"].sum()
    * 100
)

print("\nTOP 10 VENDORS BY SALES REVENUE")

print(
    vendors[
        [
            "vendor_number",
            "vendor_name",
            "revenue",
            "units_sold",
            "revenue_share"
        ]
    ]
    .head(10)
    .to_string(index=False)
)


# ============================================================
#  INVENTORY ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("9. INVENTORY ANALYSIS")
print("=" * 70)

inventory["inventory_status"] = np.select(
    [
        inventory["on_hand"] <= 0,
        inventory["on_hand"] <= 5,
        inventory["on_hand"] <= 20
    ],
    [
        "OUT_OF_STOCK",
        "CRITICAL",
        "LOW"
    ],
    default="HEALTHY"
)

inventory_summary = (
    inventory.groupby("inventory_status")
    .agg(
        products=("brand_id", "count"),
        units=("on_hand", "sum"),
        inventory_value=("inventory_value", "sum")
    )
    .reset_index()
)

print(inventory_summary.to_string(index=False))


# ============================================================
#  STOCKOUT / LOW-STOCK PRODUCTS
# ============================================================

print("\n" + "=" * 70)
print("10. STOCKOUT & LOW-STOCK PRODUCTS")
print("=" * 70)

stock_risk = (
    inventory[
        inventory["on_hand"] <= 5
    ]
    .sort_values(
        ["on_hand", "inventory_value"]
    )
)

print(
    stock_risk[
        [
            "store_id",
            "brand_id",
            "description",
            "on_hand",
            "price",
            "inventory_value"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


# ============================================================
#  SALES + INVENTORY = REORDER CANDIDATES
# ============================================================

print("\n" + "=" * 70)
print("11. REORDER CANDIDATES")
print("=" * 70)

# Product-level sales velocity
sales_velocity = (
    product_sales[
        [
            "brand_id",
            "description",
            "units_sold",
            "revenue"
        ]
    ]
    .copy()
)

# Average daily units
days = daily["sales_date"].nunique()

sales_velocity["avg_daily_units"] = (
    sales_velocity["units_sold"] / days
)

# Total inventory by product
inventory_product = (
    inventory.groupby("brand_id")
    .agg(
        inventory_units=("on_hand", "sum"),
        inventory_value=("inventory_value", "sum")
    )
    .reset_index()
)

reorder = sales_velocity.merge(
    inventory_product,
    on="brand_id",
    how="left"
)

reorder["days_of_inventory"] = np.where(
    reorder["avg_daily_units"] > 0,
    reorder["inventory_units"]
    / reorder["avg_daily_units"],
    np.inf
)

# Simple business rule:
# High sales + low inventory coverage
reorder["reorder_priority"] = np.select(
    [
        reorder["days_of_inventory"] < 7,
        reorder["days_of_inventory"] < 14,
        reorder["days_of_inventory"] < 30
    ],
    [
        "URGENT",
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

reorder_candidates = (
    reorder[
        reorder["reorder_priority"].isin(
            ["URGENT", "HIGH"]
        )
    ]
    .sort_values(
        ["reorder_priority", "units_sold"],
        ascending=[True, False]
    )
)

print(
    reorder_candidates[
        [
            "brand_id",
            "description",
            "units_sold",
            "avg_daily_units",
            "inventory_units",
            "days_of_inventory",
            "reorder_priority"
        ]
    ]
    .head(30)
    .to_string(index=False)
)


# ============================================================
#  SLOW-MOVING PRODUCTS
# ============================================================

print("\n" + "=" * 70)
print("12. SLOW-MOVING PRODUCTS")
print("=" * 70)

slow_movers = (
    reorder[
        reorder["units_sold"] <= reorder["units_sold"].quantile(0.10)
    ]
    .sort_values("units_sold")
)

print(
    slow_movers[
        [
            "brand_id",
            "description",
            "units_sold",
            "revenue",
            "inventory_units",
            "inventory_value"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


# ============================================================
#  EXCESS INVENTORY
# ============================================================

print("\n" + "=" * 70)
print("13. POTENTIAL EXCESS INVENTORY")
print("=" * 70)

excess_inventory = reorder[
    (reorder["days_of_inventory"] > 90) &
    (reorder["inventory_units"] > 0)
].sort_values(
    "inventory_value",
    ascending=False
)

print(
    excess_inventory[
        [
            "brand_id",
            "description",
            "units_sold",
            "avg_daily_units",
            "inventory_units",
            "days_of_inventory",
            "inventory_value"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


# ============================================================
#  SALES VS PURCHASE ALIGNMENT
# ============================================================

print("\n" + "=" * 70)
print("14. SALES VS PURCHASE ALIGNMENT")
print("=" * 70)

sales_purchase = (
    product_sales[
        [
            "brand_id",
            "description",
            "units_sold",
            "revenue"
        ]
    ]
    .merge(
        product_cost[
            [
                "brand_id",
                "purchased_units",
                "purchase_cost",
                "weighted_avg_cost"
            ]
        ],
        on="brand_id",
        how="left"
    )
)

sales_purchase["purchase_to_sales_ratio"] = np.where(
    sales_purchase["units_sold"] > 0,
    sales_purchase["purchased_units"]
    / sales_purchase["units_sold"],
    np.nan
)

print(
    sales_purchase.sort_values(
        "purchase_to_sales_ratio",
        ascending=False
    )[
        [
            "brand_id",
            "description",
            "units_sold",
            "purchased_units",
            "purchase_to_sales_ratio",
            "purchase_cost"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


# ============================================================
#  STORE × PRODUCT OPPORTUNITY
# ============================================================

print("\n" + "=" * 70)
print("15. STORE × PRODUCT OPPORTUNITY")
print("=" * 70)

store_product = pd.read_sql("""
    SELECT
        s.store_id,
        st.city,
        s.brand_id,
        p.description,
        SUM(s.sales_quantity) AS units_sold,
        SUM(s.sales_dollars) AS revenue
    FROM analytics.fact_sales s
    LEFT JOIN analytics.dim_store st
        ON s.store_id = st.store_id
    LEFT JOIN analytics.dim_product p
        ON s.brand_id = p.brand_id
    GROUP BY
        s.store_id,
        st.city,
        s.brand_id,
        p.description
""", engine)

print("Store-product combinations:", len(store_product))

print("\nTOP STORE × PRODUCT COMBINATIONS")

print(
    store_product
    .sort_values("revenue", ascending=False)
    .head(20)
    .to_string(index=False)
)


# ============================================================
# BEST PRODUCT FOR EACH STORE
# ============================================================

print("\n" + "=" * 70)
print("16. BEST PRODUCT PER STORE")
print("=" * 70)

store_product["rank"] = (
    store_product
    .groupby("store_id")["revenue"]
    .rank(
        method="first",
        ascending=False
    )
)

best_product_store = (
    store_product[
        store_product["rank"] == 1
    ]
    .sort_values("revenue", ascending=False)
)

print(
    best_product_store[
        [
            "store_id",
            "city",
            "brand_id",
            "description",
            "units_sold",
            "revenue"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


# ============================================================
# DAY-OF-WEEK BUSINESS ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("17. DAY-OF-WEEK PERFORMANCE")
print("=" * 70)

daily["sales_date"] = pd.to_datetime(
    daily["sales_date"]
)

daily["day_name"] = (
    daily["sales_date"]
    .dt.day_name()
)

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday = (
    daily.groupby("day_name")
    .agg(
        revenue=("revenue", "sum"),
        units=("units_sold", "sum"),
        transactions=("transaction_lines", "sum")
    )
    .reindex(weekday_order)
    .reset_index()
)

weekday["revenue_share"] = (
    weekday["revenue"]
    / weekday["revenue"].sum()
    * 100
)

print(weekday.to_string(index=False))

best_day = weekday.loc[
    weekday["revenue"].idxmax()
]

worst_day = weekday.loc[
    weekday["revenue"].idxmin()
]

print(
    f"\nBest weekday  : {best_day['day_name']} "
    f"(${best_day['revenue']:,.2f})"
)

print(
    f"Worst weekday : {worst_day['day_name']} "
    f"(${worst_day['revenue']:,.2f})"
)


# ============================================================
# BUSINESS OPPORTUNITY SCORING
# ============================================================

print("\n" + "=" * 70)
print("18. BUSINESS OPPORTUNITY SCORING")
print("=" * 70)

opportunity = reorder.copy()

# Normalize sales and inventory risk
opportunity["sales_score"] = (
    opportunity["units_sold"]
    / opportunity["units_sold"].max()
)

opportunity["inventory_risk"] = np.where(
    opportunity["days_of_inventory"] < 14,
    1,
    0
)

opportunity["slow_inventory_risk"] = np.where(
    opportunity["days_of_inventory"] > 90,
    1,
    0
)

opportunity["opportunity_score"] = (
    opportunity["sales_score"] * 0.6
    + opportunity["inventory_risk"] * 0.4
)

print("\nTOP BUSINESS OPPORTUNITIES")

print(
    opportunity.sort_values(
        "opportunity_score",
        ascending=False
    )[
        [
            "brand_id",
            "description",
            "units_sold",
            "inventory_units",
            "days_of_inventory",
            "opportunity_score"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


# ============================================================
#  AUTOMATIC BUSINESS RECOMMENDATIONS
# ============================================================

print("\n" + "=" * 70)
print("19. AUTOMATIC BUSINESS RECOMMENDATIONS")
print("=" * 70)

recommendations = []

# Product concentration
top_20_revenue = (
    product
    .head(20)["revenue"]
    .sum()
    / product["revenue"].sum()
    * 100
)

recommendations.append(
    f"Top 20 products generate approximately "
    f"{top_20_revenue:.1f}% of total revenue."
)

# Best store
top_store = stores.iloc[0]

recommendations.append(
    f"Store {int(top_store['store_id'])} "
    f"({top_store['city']}) is the highest-revenue store."
)

# Best vendor
top_vendor = vendors.iloc[0]

recommendations.append(
    f"Vendor {int(top_vendor['vendor_number'])} "
    f"({top_vendor['vendor_name']}) has the highest "
    f"associated sales revenue."
)

# Stockout risk
urgent_count = (
    reorder["reorder_priority"] == "URGENT"
).sum()

recommendations.append(
    f"{urgent_count:,} product records have less than "
    f"7 days of estimated inventory coverage."
)

# Excess inventory
excess_count = len(excess_inventory)

recommendations.append(
    f"{excess_count:,} products have more than "
    f"90 days of estimated inventory coverage and "
    f"may require inventory reduction or promotion."
)

# Best weekday
recommendations.append(
    f"{best_day['day_name']} is the strongest sales weekday."
)

# Slow movers
recommendations.append(
    f"{len(slow_movers):,} products fall into the "
    f"lowest 10% of unit sales and should be reviewed "
    f"for assortment optimization."
)

for i, recommendation in enumerate(
    recommendations,
    start=1
):
    print(f"{i}. {recommendation}")


# ============================================================
# BUSINESS ACTION PLAN
# ============================================================

print("\n" + "=" * 70)
print("20. MANAGEMENT ACTION PLAN")
print("=" * 70)

action_plan = pd.DataFrame({
    "Area": [
        "High-performing products",
        "Low-stock products",
        "Excess inventory",
        "Low-margin products",
        "Top stores",
        "Weak stores",
        "Vendor management",
        "Slow-moving products"
    ],

    "Recommended_Action": [
        "Maintain availability and prioritize these products.",
        "Increase reorder priority for high-demand products.",
        "Reduce purchasing or use promotions to clear excess stock.",
        "Review supplier cost, selling price and pricing strategy.",
        "Study successful store/product combinations.",
        "Investigate demand, assortment and inventory differences.",
        "Compare vendors using purchase cost and sales contribution.",
        "Review assortment and consider discounts or discontinuation."
    ]
})

print(action_plan.to_string(index=False))


# ============================================================
# 21. EXPORT BUSINESS ANALYTICS
# ============================================================

print("\n" + "=" * 70)
print("21. EXPORTING RESULTS")
print("=" * 70)

export_path = r"D:\data\business_analytics"

import os
os.makedirs(export_path, exist_ok=True)

product.to_csv(
    f"{export_path}\\product_performance.csv",
    index=False
)

abc.to_csv(
    f"{export_path}\\product_abc_analysis.csv",
    index=False
)

margin.to_csv(
    f"{export_path}\\product_margin_analysis.csv",
    index=False
)

stores.to_csv(
    f"{export_path}\\store_performance.csv",
    index=False
)

vendors.to_csv(
    f"{export_path}\\vendor_performance.csv",
    index=False
)

inventory_summary.to_csv(
    f"{export_path}\\inventory_summary.csv",
    index=False
)

reorder_candidates.to_csv(
    f"{export_path}\\reorder_candidates.csv",
    index=False
)

slow_movers.to_csv(
    f"{export_path}\\slow_moving_products.csv",
    index=False
)

excess_inventory.to_csv(
    f"{export_path}\\excess_inventory.csv",
    index=False
)

store_product.to_csv(
    f"{export_path}\\store_product_performance.csv",
    index=False
)

weekday.to_csv(
    f"{export_path}\\weekday_performance.csv",
    index=False
)

action_plan.to_csv(
    f"{export_path}\\management_action_plan.csv",
    index=False
)

print("Business analytics exported to:")
print(export_path)


